In [1]:
import os
import pandas as pd
import vnstock
from dotenv import load_dotenv

# Setup

In [2]:
load_dotenv()
vnstock_id = os.getenv("VNSTOCK_ID")
if not vnstock_id:
    raise ValueError("VNSTOCK_ID is not set. Add it to .env")

vnstock.login(vnstock_id);

In [3]:
account_info = vnstock.status()
print(f"Account authenticated status: {account_info['authenticated']}")
print(f"Account tier status: {account_info['tier']}")

Account authenticated status: True
Account tier status: free


## 1.1 Listing

In [4]:
ticker_table = vnstock.Reference().equity.list_by_exchange()

In [5]:
#number of rows and number of unique symbols
print(f"number of rows: {len(ticker_table)}")
print(f"number of unique symbols: {ticker_table['symbol'].nunique()}")

number of rows: 1755
number of unique symbols: 1755


- `symbol` is unique, no NULL symbols

In [6]:
# table information
ticker_table.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 1755 entries, 0 to 1754
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   symbol            1755 non-null   string
 1   exchange          1755 non-null   string
 2   organ_name        1728 non-null   string
 3   organ_short_name  1728 non-null   string
 4   icb_code_lv2      1736 non-null   string
dtypes: string(5)
memory usage: 681.6 KB


In [7]:
ticker_table[ticker_table["organ_name"].isna() == True]

,symbol,exchange,organ_name,organ_short_name,icb_code_lv2
131,VHI,DELISTED,<NA>,<NA>,3500
265,TS5,DELISTED,<NA>,<NA>,<NA>
392,T12,DELISTED,<NA>,<NA>,<NA>
401,SVL,DELISTED,<NA>,<NA>,2700
444,SON,DELISTED,<NA>,<NA>,2700
724,PDT,DELISTED,<NA>,<NA>,7500
788,NNQ,DELISTED,<NA>,<NA>,3500
834,MXC,DELISTED,<NA>,<NA>,<NA>
1020,IPH,DELISTED,<NA>,<NA>,<NA>
1056,HUX,DELISTED,<NA>,<NA>,<NA>


In [8]:
ticker_table[ticker_table["icb_code_lv2"].isna()]

,symbol,exchange,organ_name,organ_short_name,icb_code_lv2
265,TS5,DELISTED,<NA>,<NA>,<NA>
387,TAC,DELISTED,Công ty Cổ phần Dầu Thực vật Tường An,Dầu Tường An,<NA>
392,T12,DELISTED,<NA>,<NA>,<NA>
471,SIC,DELISTED,Công ty Cổ phần ANI,Công ty Cổ phần ANI,<NA>
598,PYU,DELISTED,Công ty Cổ phần Môi trường và Công trình đô th...,Công trình đô thị Phúc Yên,<NA>
834,MXC,DELISTED,<NA>,<NA>,<NA>
1020,IPH,DELISTED,<NA>,<NA>,<NA>
1056,HUX,DELISTED,<NA>,<NA>,<NA>
1151,HGR,DELISTED,<NA>,<NA>,<NA>
1153,HGC,DELISTED,<NA>,<NA>,<NA>


In [9]:
# total ticker per exchange
ticker_table.groupby("exchange").agg({"symbol": "count"})

,symbol
exchange,
DELISTED,238
HNX,299
HOSE,408
UPCOM,810


In [10]:
ticker_table[ticker_table["symbol"].str.len() != 3]

,symbol,exchange,organ_name,organ_short_name,icb_code_lv2


- `organ_name` and `organ_short_name` columns are string columns with both have 27 NULL values.
- `icb_code_lv2` column is a string columns that has 19 NULL values.
- There are 3 different exchanges: HNX, HOSE, UPCOM.
- HNX has 299 active tickers, HOSE has 408 tickers, and UPCOM has 810 tickers.
- DELISTED refer to stocks that were previously listed on one of the three exchanges but are no longer traded.
- All NULL values of `organ_name` and `organ_short_name` are from DELISTED ticker.
- All NULL values of `icb_code_lv2` are from DELISTED ticker.
- All symbols have exactly 3 characters.

## 1.2 OHLCV

Default source for `ohlcv()` is VCI (same as the listing). The pipeline will pass `source` explicitly.

In [11]:
mkt = vnstock.Market()
fpt_ohlcv = mkt.equity("FPT").ohlcv(start="2013-05-22")

c:\Users\Long\Desktop\flagships\.venv\Lib\site-packages\vnstock\ui\market.py:280: UserWarning: This result is limited to what the Community tier includes: the latest 2,920 days of history only. 1,366 rows were left out. Fuller coverage is part of the usage-rights plans: https://vnstocks.com/pricing
  bars = self._call(


In [12]:
fpt_ohlcv

,time,open,high,low,close,volume
0,2018-11-06 07:00:00+07:00,12.09,12.11,11.92,11.94,470930
1,2018-11-07 07:00:00+07:00,11.94,12.03,11.79,11.85,495560
2,2018-11-08 07:00:00+07:00,11.96,12.01,11.88,11.94,352130
3,2018-11-09 07:00:00+07:00,11.84,11.92,11.67,11.68,635120
4,2018-11-12 07:00:00+07:00,11.67,11.88,11.65,11.88,273110
...,...,...,...,...,...,...
1990,2026-10-29 07:00:00+07:00,54.50,54.80,52.50,52.70,12591100
1991,2026-10-30 07:00:00+07:00,52.40,53.70,52.40,52.70,5661600
1992,2026-11-02 07:00:00+07:00,52.40,52.50,50.10,50.20,12293400
1993,2026-11-03 07:00:00+07:00,50.30,51.70,49.55,50.40,9249300


- ohlcv data goes back 2,920 days = 1,995 trading rows (community tier limit)

In [13]:
fpt_ohlcv.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 1995 entries, 0 to 1994
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype                           
---  ------  --------------  -----                           
 0   time    1995 non-null   datetime64[ns, Asia/Ho_Chi_Minh]
 1   open    1995 non-null   float64                         
 2   high    1995 non-null   float64                         
 3   low     1995 non-null   float64                         
 4   close   1995 non-null   float64                         
 5   volume  1995 non-null   int64                           
dtypes: datetime64[ns, Asia/Ho_Chi_Minh](1), float64(4), int64(1)
memory usage: 93.6 KB


- currency unit is thousands of VND/Vietnam Dong (₫)
- the data in ohlcv is "adjusted Data", the old data has odd decimal but new data has clean decimal because company undergoes a split or issues a dividend. 
- 0.047 kb or 47 b a row for one ticker. BigQuery also needs a `symbol` column, so rows will be bigger.

In [14]:
fpt_ohlcv["time"].dt.time.nunique()

1

- The clock part is always 07:00 (+7:00) = 00:00 UTC -> the column means a trading date -> DATE in BigQuery.

In [15]:
fpt_ohlcv["time"].nunique()

1995

- fpt_ohlcv table has 1995 rows, column `time` has 1995 unique values -> `time` is unique.

In [16]:
# Tet 2026 period
fpt_ohlcv[(fpt_ohlcv["time"] >= "2026-02-08") & (fpt_ohlcv["time"] <= "2026-02-27")]

,time,open,high,low,close,volume
1813,2026-02-09 07:00:00+07:00,88.60,89.14,87.79,88.78,5123453
1814,2026-02-10 07:00:00+07:00,89.58,89.58,87.25,87.70,6143120
1815,2026-02-11 07:00:00+07:00,88.06,89.23,87.34,88.42,5525584
1816,2026-02-12 07:00:00+07:00,88.78,88.96,88.24,88.60,4137993
1817,2026-02-13 07:00:00+07:00,87.97,88.33,86.36,86.36,18546292
1818,2026-02-23 07:00:00+07:00,86.80,86.80,83.49,84.74,22701438
1819,2026-02-24 07:00:00+07:00,84.29,84.29,82.95,82.95,28627330
1820,2026-02-25 07:00:00+07:00,82.68,82.95,79.81,79.90,47840750
1821,2026-02-26 07:00:00+07:00,80.71,82.68,80.62,81.15,17593368


- Holidays in Vietnam result in missing rows for the days on which they fall.

In [17]:
fpt_ohlcv[fpt_ohlcv["volume"] == 0]

,time,open,high,low,close,volume


- No `volume` = 0. Check again with smaller ticker in 1.3

open questions:
- adjusted prices + rolling window (3.2)
- 1 machine -> self-hosted runner (5.1)